# Reservoir Sampling

## Learning Objectives

1. **State** the uniform sampling problem for streams of unknown length
2. **Prove** that reservoir sampling maintains a uniform sample at all times
3. **Implement** Algorithm R and weighted reservoir sampling
4. **Extend** to sampling $k$ elements uniformly from a stream

## The Problem

**Goal:** maintain a uniform random sample of size $k$ from a stream $a_1, a_2, \ldots, a_n$ where $n$ is **unknown**.

**Constraint:** only one pass; cannot store all elements.

**Simple case $k=1$:** maintain one element; replace it with probability $1/t$ when seeing the $t$-th element.

**Claim:** after seeing $t$ elements, every element has been selected with probability $1/t$.

**Proof by induction:** base case $t=1$: trivially true.  
Inductive step: element $i$ (for $i < t$) is in reservoir with prob $1/(t-1)$.  
It survives step $t$ with prob $1 - 1/t = (t-1)/t$.  
Combined: $\frac{1}{t-1} \cdot \frac{t-1}{t} = \frac{1}{t}$. ✓

In [1]:
import random, math

def reservoir_sample_k1(stream, seed=0):
    """Sample exactly 1 element uniformly from stream."""
    ""
    rng = random.Random(seed)
    reservoir = None
    for t, item in enumerate(stream, start=1):
        if rng.random() < 1/t:
            reservoir = item
    return reservoir

# Verify uniformity
stream = list(range(100))
from collections import defaultdict
counts = defaultdict(int)
n_trials = 100000
for trial in range(n_trials):
    chosen = reservoir_sample_k1(stream, seed=trial)
    counts[chosen] += 1

# Check uniformity
expected = n_trials / len(stream)
min_f = min(counts.values()) / n_trials
max_f = max(counts.values()) / n_trials
print(f"Expected frequency per item: {1/len(stream):.4f}")
print(f"Observed range: [{min_f:.4f}, {max_f:.4f}]")
print(f"Uniform: {abs(min_f - max_f) < 0.005}")

Expected frequency per item: 0.0100
Observed range: [0.0091, 0.0109]
Uniform: True


In [2]:
def reservoir_sample_k(stream, k, seed=0):
    """Algorithm R: sample k elements uniformly from a stream."""
    ""
    rng = random.Random(seed)
    reservoir = []
    for t, item in enumerate(stream, start=1):
        if t <= k:
            reservoir.append(item)
        else:
            # Replace reservoir[j] with probability k/t
            j = rng.randint(0, t-1)
            if j < k:
                reservoir[j] = item
    return reservoir

# Test
stream = list(range(1000))
k = 5
counts_k = defaultdict(int)
n_trials = 50000
for trial in range(n_trials):
    sample = reservoir_sample_k(stream, k, seed=trial)
    for item in sample:
        counts_k[item] += 1

expected_k = n_trials * k / len(stream)
obs_vals = list(counts_k.values())
print(f"k={k}, stream size=1000")
print(f"Expected count per item: {expected_k:.1f}")
print(f"Observed: mean={sum(obs_vals)/len(obs_vals):.1f}, "
      f"min={min(obs_vals)}, max={max(obs_vals)}")
print(f"Coefficient of variation: {(max(obs_vals)-min(obs_vals))/(2*expected_k):.3f}")

k=5, stream size=1000
Expected count per item: 250.0
Observed: mean=250.0, min=201, max=298
Coefficient of variation: 0.194


## Weighted Reservoir Sampling

When items have weights $w_i$ and we want each item sampled proportional to $w_i$:

**Efraimidis-Spirakis (2006):**
For each item $i$ with weight $w_i$, assign key $u_i^{1/w_i}$ where $u_i \sim \text{Uniform}(0,1)$.
Keep the $k$ items with the largest keys.

This is equivalent to assigning each item a random priority proportional to its weight.

In [3]:
import heapq

def weighted_reservoir_k(stream_with_weights, k, seed=0):
    """Weighted reservoir sampling (Efraimidis-Spirakis algorithm)."""
    ""
    rng = random.Random(seed)
    # Min-heap of (key, item)
    heap = []
    for item, weight in stream_with_weights:
        u = rng.random()
        key = u ** (1.0/weight)
        if len(heap) < k:
            heapq.heappush(heap, (key, item))
        elif key > heap[0][0]:
            heapq.heapreplace(heap, (key, item))
    return [item for key, item in heap]

# Test: item 0 has weight 10, items 1-9 have weight 1
items_weights = [(0, 10)] + [(i, 1) for i in range(1,10)]
counts_w = defaultdict(int)
k = 3
for trial in range(50000):
    sample = weighted_reservoir_k(items_weights, k, seed=trial)
    for s in sample: counts_w[s] += 1

total_weight = 10 + 9
print("Weighted reservoir sampling (item 0 has 10x weight):")
for item in range(10):
    w = 10 if item==0 else 1
    expected_frac = w/total_weight
    obs_frac = counts_w[item]/50000/k*len(items_weights)
    print(f"  item {item}: expected_frac={expected_frac:.3f}, observed={counts_w[item]/50000:.3f}")
    if item > 2: break

Weighted reservoir sampling (item 0 has 10x weight):
  item 0: expected_frac=0.526, observed=0.913
  item 1: expected_frac=0.053, observed=0.230
  item 2: expected_frac=0.053, observed=0.231
  item 3: expected_frac=0.053, observed=0.231
